In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 11 — Ejercicio 1
# ---------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

california = fetch_california_housing()
X, y = california.data, california.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lrs = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
rmse_lr = []

for lr in lrs:
    model = XGBRegressor(
        n_estimators=300, learning_rate=lr,
        max_depth=4, random_state=42,
        eval_metric="rmse", verbosity=0
    )
    model.fit(X_train, y_train)
    rmse = np.sqrt(mean_squared_error(
        y_test, model.predict(X_test)))
    rmse_lr.append(rmse)
    print(f"lr={lr} | RMSE={rmse:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lrs, rmse_lr, marker="o")
ax.set_xscale("log")
ax.set_xlabel("learning_rate (escala log)")
ax.set_ylabel("RMSE en test")
ax.set_title("XGBoost — RMSE vs learning_rate (n_est=300)")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 11 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import time

# Subconjunto para agilizar la búsqueda
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=5000, replace=False)
X_search = X_train[idx]
y_search = y_train[idx]

param_dist = {
    "n_estimators":  randint(100, 400),
    "max_depth":     randint(3, 7),
    "learning_rate": uniform(0.01, 0.2),
    "subsample":     uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.5, 0.5),
}

rs = RandomizedSearchCV(
    XGBRegressor(eval_metric="rmse", verbosity=0, random_state=42),
    param_dist,
    n_iter=15, cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1
)

t0 = time.time()
rs.fit(X_search, y_search)
print(f"Tiempo: {time.time()-t0:.1f}s")
print(f"Mejores params: {rs.best_params_}")
rmse_rs = np.sqrt(mean_squared_error(
    y_test, rs.best_estimator_.predict(X_test)))
print(f"RMSE en test: {rmse_rs:.4f}")

rmse_es_paso4 = 0.4389  # Valor obtenido en el Paso 4
print(f"\nComparación:")
print(f"XGBoost + Early Stopping (Paso 4): RMSE = {rmse_es_paso4:.4f}")
print(f"XGBoost + RandomizedSearchCV (Ej 2): RMSE = {rmse_rs:.4f}")
print("Conclusión: En esta configuración, el Early Stopping con más árboles y validación directa")
print("superó a la búsqueda aleatoria con 15 iteraciones y submuestreo.")

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 11 — Ejercicio 3
# ---------------------------------------------------------------
# r2_score se encuenta definido en el capítulo 11

import time
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import lightgbm as lgb

# 1. Entrenar todos los modelos y guardar resultados
resultados = {}

# Lineal
t0 = time.time()
lr = LinearRegression().fit(X_train, y_train)
pred = lr.predict(X_test)
resultados["Lineal"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred),
    "Fit(s)": time.time() - t0
}

# Random Forest
t0 = time.time()
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train, y_train)
pred = rf.predict(X_test)
resultados["Random Forest"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred),
    "Fit(s)": time.time() - t0
}

# GB sklearn
t0 = time.time()
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42).fit(X_train, y_train)
pred = gb.predict(X_test)
resultados["GB sklearn"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred),
    "Fit(s)": time.time() - t0
}

# XGBoost (reentrenado rápido para consistencia en la tabla)
t0 = time.time()
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42, n_jobs=-1, verbosity=0).fit(X_train, y_train)
pred = xgb.predict(X_test)
resultados["XGBoost"] = {
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred),
    "Fit(s)": time.time() - t0
}

# LightGBM (tu modelo ya entrenado)
resultados["LightGBM"] = {
    "RMSE": rmse_lgbm,
    "R2": r2_lgbm,
    "Fit(s)": t_lgbm
}

# 2. Crear y mostrar la tabla
df_comp = pd.DataFrame(resultados).T.round(4)
print(df_comp)
